In [18]:
import pandas as pd
import datetime

from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/My Drive/Learn/Associate Data Scientist/Fundamental of Deep Learning - Nasional/nba2k20-full.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 429 entries, 0 to 428
Data columns (total 14 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   full_name    429 non-null    object
 1   rating       429 non-null    int64 
 2   jersey       429 non-null    object
 3   team         406 non-null    object
 4   position     429 non-null    object
 5   b_day        429 non-null    object
 6   height       429 non-null    object
 7   weight       429 non-null    object
 8   salary       429 non-null    object
 9   country      429 non-null    object
 10  draft_year   429 non-null    int64 
 11  draft_round  429 non-null    object
 12  draft_peak   429 non-null    object
 13  college      363 non-null    object
dtypes: int64(2), object(12)
memory usage: 47.1+ KB


In [20]:
def from_date_to_age(date):
    born=datetime.datetime.strptime(date, '%m/%d/%y')
    today = datetime.date.today()
    return today.year - born.year - ((today.month, today.day) < (born.month, born.day))
def int_weight(weight):
    return weight.split("/")[1].split(" ")[1]
def int_height(height):
    return height.split("/")[1].split(" ")[1]

df['height']=df['height'].apply(lambda x: int_height(x)).astype('float')
df['weight']=df['weight'].apply(lambda x: int_weight(x)).astype('float')

# convert height and weight to float values and rename
df.rename({'height':'height_in_m','weight':'weight_in_kg'},axis='columns',inplace=True)

# convert salary to int values
df['salary'] = df['salary'].str[1:].astype('int64')

# convert draft round,pick to int and handle missing value
df['draft_round'] = df['draft_round'].replace({'Undrafted': 0}).astype('int8')

df['jersey'] = df['jersey'].str[1:].astype('int8')

# Indicate if a nba players attended college
df['college'].isna().astype('int').value_counts()
df['attended_college'] = ~(df['college'].isna().astype('bool'))

# Indicate their current age
df['current_age']=df['b_day'].apply(lambda x: from_date_to_age(x))

# indicate the numbers of years played since they started nba
df['year_played'] = df['current_age'] - (df['draft_year'] - pd.to_datetime(df['b_day']).dt.year)
df.drop(columns=['b_day'],inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 429 entries, 0 to 428
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   full_name         429 non-null    object 
 1   rating            429 non-null    int64  
 2   jersey            429 non-null    int8   
 3   team              406 non-null    object 
 4   position          429 non-null    object 
 5   height_in_m       429 non-null    float64
 6   weight_in_kg      429 non-null    float64
 7   salary            429 non-null    int64  
 8   country           429 non-null    object 
 9   draft_year        429 non-null    int64  
 10  draft_round       429 non-null    int8   
 11  draft_peak        429 non-null    object 
 12  college           363 non-null    object 
 13  attended_college  429 non-null    bool   
 14  current_age       429 non-null    int64  
 15  year_played       429 non-null    int64  
dtypes: bool(1), float64(2), int64(5), int8(2), o

/tmp/ipykernel_302/2781662472.py:32: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['year_played'] = df['current_age'] - (df['draft_year'] - pd.to_datetime(df['b_day']).dt.year)


In [21]:
data_pokok = df[['rating', 'height_in_m', 'weight_in_kg',
'current_age', 'year_played']]

In [22]:
data_label = df['salary']

In [24]:
from sklearn.model_selection import train_test_split

data_pokok_train, data_pokok_test, data_label_train, data_label_test = train_test_split(
data_pokok, data_label, test_size=0.2, random_state=42)

In [25]:
from sklearn.linear_model import LinearRegression, ElasticNetCV
model1 = LinearRegression() #buat model regresi linier
model2 = ElasticNetCV( #buat model ElasticNet
l1_ratio=[.1, .2, .3, .4, .5, .6, .7, .8, .9, 1.],
cv=5)

In [26]:
model1.fit(data_pokok_train, data_label_train)
model2.fit(data_pokok_train, data_label_train)

ElasticNetCV(cv=5, l1_ratio=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0])

In [27]:
from sklearn.metrics import mean_absolute_error
#prediksi semua data dengan model1
prediksi1 = model1.predict(data_pokok_test)
#hitung MAE dari hasil prediksi dengan model1
MAE1 = mean_absolute_error(data_label_test, prediksi1)
print(f'Model1: {MAE1}')
#prediksi semua data dengan model2
prediksi2 = model2.predict(data_pokok_test)
#hitung MAE dari hasil prediksi dengan model2
MAE2 = mean_absolute_error(data_label_test, prediksi2)
print(f'Model2: {MAE2}')

Model1: 4513699.99379235
Model2: 4463376.213679069


In [28]:
model2.fit(data_pokok, data_label)

ElasticNetCV(cv=5, l1_ratio=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0])

In [29]:
model2.coef_

array([1134989.62574992,       0.        ,  -38527.79612013,
        266666.26049746,  305872.29978645])

In [30]:
model2.intercept_

np.float64(-86346992.18770784)

In [31]:
pd.DataFrame(data=model2.coef_, index=data_pokok.columns)

,0
rating,1.134990e+06
height_in_m,0.000000e+00
weight_in_kg,-3.852780e+04
current_age,2.666663e+05
year_played,3.058723e+05


In [32]:
#gaji aktual 2020
gaji2020 = 37436858

#kondisi atribut data LeBron James tahun 2020
#rating, tinggi, berat, umur dan pengalaman
leBron2020 = [[97, 2.06, 113.4, 36, 17]]

# prediksi gaji di 2020
est2020 = model2.predict(leBron2020)
print(f'estimasi gaji: {est2020} dengan kesalahan estimasi: {est2020-gaji2020}')

estimasi gaji: [34177763.90428925] dengan kesalahan estimasi: [-3259094.09571075]


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but ElasticNetCV was fitted with feature names
  warnings.warn(


In [33]:
# Kondisi atribut data LeBron James tahun 2024
# Rating, tinggi, berat, umur dan pengalaman
leBron2024 = [[97, 2.06, 113.4, 40, 21]]
# prediksi gaji di 2024
est2024 = model2.predict(leBron2024)
print(est2024)

[36467918.14542489]


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but ElasticNetCV was fitted with feature names
  warnings.warn(
